<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/Gradient_boosting_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install if needed (Colab)
import subprocess, sys
for pkg in ['xgboost', 'lightgbm', 'catboost']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               VotingClassifier, StackingClassifier,GradientBoostingRegressor)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('All imports successful.')
print(f'  XGBoost  : {xgb.__version__}')
print(f'  LightGBM : {lgb.__version__}')
print(f'  CatBoost : {cb.__version__}')

All imports successful.
  XGBoost  : 3.4.1
  LightGBM : 4.6.0
  CatBoost : 1.2.10


In [ ]:
#creating data frame out of csv file
loan_data= pd.read_csv("/content/loan_default_data_1.csv")

In [ ]:
#verifying whether the data has been uploaded
loan_data.head(10)

,loan_amount,rate_of_interest,credit_score,income,months_employed,num_loans,dependants,default
0,8000,6.5,740,72000,60,1,0,0
1,10000,7.2,720,65000,48,1,1,0
2,12000,8.1,695,55000,36,2,0,0
3,14000,9.0,675,50000,30,2,1,0
4,11000,7.8,710,62000,42,1,0,0
5,9500,7.0,730,68000,55,1,0,0
6,30000,15.5,520,28000,3,7,3,1
7,28000,14.0,540,31000,5,6,2,1
8,25000,13.2,555,34000,7,5,3,1
9,32000,16.0,510,25000,2,8,4,1


In [ ]:
#getting the information of the dataset
loan_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   loan_amount       12 non-null     int64  
 1   rate_of_interest  12 non-null     float64
 2   credit_score      12 non-null     int64  
 3   income            12 non-null     int64  
 4   months_employed   12 non-null     int64  
 5   num_loans         12 non-null     int64  
 6   dependants        12 non-null     int64  
 7   default           12 non-null     int64  
dtypes: float64(1), int64(7)
memory usage: 900.0 bytes


In [ ]:
#getting stats of the data
loan_data.describe()

,loan_amount,rate_of_interest,credit_score,income,months_employed,num_loans,dependants,default
count,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000
mean,19375.000000,11.133333,622.083333,45750.000000,24.833333,3.916667,1.583333,0.500000
std,9286.170069,3.800797,95.547664,17970.303787,22.618710,2.810963,1.443376,0.522233
min,8000.000000,6.500000,510.000000,25000.000000,2.000000,1.000000,0.000000,0.000000
25%,10750.000000,7.650000,537.500000,29750.000000,4.750000,1.000000,0.000000,0.000000
50%,19500.000000,11.100000,615.000000,42000.000000,18.500000,3.500000,1.500000,0.500000
75%,27250.000000,14.575000,712.500000,62750.000000,43.500000,6.250000,3.000000,1.000000
max,32000.000000,16.000000,740.000000,72000.000000,60.000000,8.000000,4.000000,1.000000


In [ ]:
#splitting data into features and targets
X_loan=loan_data[["loan_amount", "rate_of_interest", "credit_score", "income", "months_employed", "num_loans","dependants"]]
y_loan=loan_data["default"]

In [ ]:
#Splitting data into the train and test
X_train, X_test, y_train, y_test = train_test_split(X_loan, y_loan, test_size=0.2, random_state=42,stratify=y_loan)

In [ ]:
#XGboost-- using Dmatrix for better performance
dtrain=xgb.DMatrix(X_train, label=y_train)
dtest=xgb.DMatrix(X_test, label=y_test)

In [ ]:
#training the model with regularization and without regularization.
configs={
          'No Regularization(Model A)': dict(max_depth=3, reg_lambda=0, gamma=0, subsample=1.0,
                                    colsample_bytree=1.0, learning_rate=0.3),
          'Regularization(Model B)': dict(max_depth=3, reg_lambda=2, gamma=1.0, subsample=0.8,
                                    colsample_bytree=0.8, learning_rate=0.1)
}

print(f"{"-"*70}")
print(f"{"Configuration":<22} {"Train accuracy":>10} {"Test accuracy":>10} {"AUC value":>10} {"Gap":>8} {"Gain":>10}")
print(f"{"-"*70}")

for name, reg_params in configs.items():
  clf=xgb.XGBClassifier(n_estimators=50, random_state=42, eval_metric="logloss", min_child_weight=0.001 ,verbosity=0, use_label_encoder=False, **reg_params)
  clf.fit(X_train, y_train)
  train_acc=clf.score(X_train, y_train)
  test_acc=clf.score(X_test, y_test)
  y_pred=clf.predict(X_test)
  auc_value=roc_auc_score(y_test, y_pred)
  gain_score=clf.get_booster().get_score(importance_type='gain')
  print(f"{name:<22} {train_acc:>10.4f} {test_acc:>10.4f} {auc_value:>10.4f} {train_acc-test_acc:>8.4f} {sum(gain_score.values()):>10.4f}")

----------------------------------------------------------------------
Configuration          Train accuracy Test accuracy  AUC value      Gap       Gain
----------------------------------------------------------------------
No Regularization(Model A)     1.0000     1.0000     1.0000   0.0000     0.9245
Regularization(Model B)     1.0000     1.0000     1.0000   0.0000     5.0665
